In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
from sentence_transformers import util
import random
import yaml

PROJECT_ROOT = Path.cwd()
with open(PROJECT_ROOT / "config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

paths = config["paths"]
evaluation_config = config["evaluation"]
random_seed = config["project"]["random_seed"]

# Load dataset
df = pd.read_csv(PROJECT_ROOT / paths["bertopic_ready"])

# Load embeddings
embeddings = np.load(PROJECT_ROOT / paths["embeddings"])

d:\bertopic_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Check embedding shape
print(df.shape)
print(embeddings.shape)

(35993, 18)
(35993, 2048)


In [3]:
#see floating-point values
print(embeddings[0][:10])

[-0.02588   -0.0004673  0.00198    0.02971   -0.00821    0.004948
 -0.00213    0.01443   -0.01628   -0.004116 ]


"Do two complaints that should be similar actually have a high similarity score?"

In [4]:
def compare_complaints(idx1, idx2):

    similarity = util.cos_sim(
        embeddings[idx1],
        embeddings[idx2]
    ).item()

    print("Similarity:", round(similarity,4))

    print("\nComplaint 1:")
    print(
        df.iloc[idx1]["Consumer complaint narrative"][:500]#first 500 characters
    )

    print("\nComplaint 2:")
    print(
        df.iloc[idx2]["Consumer complaint narrative"][:500]
    )

In [5]:
compare_complaints(1, 50)

Similarity: 0.2378

Complaint 1:
on Monday, [PROTECTED]/[PROTECTED]/[PROTECTED] of [PROTECTED] I sent a dispute to Cash app about four charges of the amount of {$83.00} that was taken out of my account on different dateson [PROTECTED]/[PROTECTED]/[PROTECTED] for some reason I was reimbursed three out of the four charges That I was disputing and I contacted Cash app about the reason for the denial on one of them and they wouldnt give me a proper reasoning and I sent them proof that the charges were being taken out with consents 

Complaint 2:
The Credit bereaus stated Account Name & Account # was properly investigation but how is that possible if the open date is inaccurate, the date last active, and date last reported is not accurate. This ground for removal. They also violated my right under 15 U.S.C 1681 Section 604 A Section 2 : The law clearly states a consumer reporting agency can not furnish a account withouth my written instructions. I haven't given these accounts written permis

randomly takes 100 complaints, compares every complaint with every other complaint, then shows:

The most similar pair â†’ to check if embeddings put similar complaints close.
The least similar pair â†’ to check if embeddings separate unrelated complaints.

In [6]:
# -----------------------------
# Sample random complaints
# -----------------------------
random.seed(random_seed)

sample_size = evaluation_config["embedding_test_sample_size"]
sample_idx = random.sample(range(len(df)), sample_size)

sample_embeddings = embeddings[sample_idx]

# Cosine similarity matrix
sim_matrix = util.cos_sim(sample_embeddings, sample_embeddings).numpy()

# Ignore self-similarity
np.fill_diagonal(sim_matrix, -1)

# -----------------------------
# Most similar pair
# -----------------------------
max_pos = np.unravel_index(np.argmax(sim_matrix), sim_matrix.shape)

idx1 = sample_idx[max_pos[0]]
idx2 = sample_idx[max_pos[1]]

print("="*80)
print("MOST SIMILAR PAIR")
print("="*80)

print(f"Similarity: {sim_matrix[max_pos]:.4f}\n")

print("Complaint 1:\n")
print(df.iloc[idx1]["Consumer complaint narrative"])

print("\n" + "-"*80 + "\n")

print("Complaint 2:\n")
print(df.iloc[idx2]["Consumer complaint narrative"])

# -----------------------------
# Least similar pair
# -----------------------------
np.fill_diagonal(sim_matrix, 2)

min_pos = np.unravel_index(np.argmin(sim_matrix), sim_matrix.shape)

idx1 = sample_idx[min_pos[0]]
idx2 = sample_idx[min_pos[1]]

print("\n\n")
print("="*80)
print("LEAST SIMILAR PAIR")
print("="*80)

print(f"Similarity: {sim_matrix[min_pos]:.4f}\n")

print("Complaint 1:\n")
print(df.iloc[idx1]["Consumer complaint narrative"])

print("\n" + "-"*80 + "\n")

print("Complaint 2:\n")
print(df.iloc[idx2]["Consumer complaint narrative"])

MOST SIMILAR PAIR
Similarity: 0.9341

Complaint 1:

I am filing a complaint against cash app ( Block , Inc. ) due to inadequate customer service and unfair practices which violate the consumer financial protection act. ( CFPA ) Specifically Cash App failed to take timely and effective measures to prevent and address fraud on their platform, leaving my account vulnerable and unprotected. Furthermore, their dispute resolution process was unfair and deceptive, as they did not comply with error resolution requirements under the Electronic Fund Transfer Act ( EFTA ) and Regulation E. These actions have caused significant inconvenience and financial loss, and I seek appropriate redress for these violations.

--------------------------------------------------------------------------------

Complaint 2:

I am following a compliant against cash app ( block, inc. ) due to inadequate customer service and unfair practices, which violates the customer financial protection act ( CFPA ). Specifically

Instead of manually inspecting two complaints, it asks:

Among the most similar complaint pairs, how often do they belong to the same Product or Issue?

In [7]:
import numpy as np
import random
from sentence_transformers import util

random.seed(random_seed)

sample_size = evaluation_config["embedding_pair_sample_size"]
sample_idx = random.sample(range(len(df)), sample_size)

sample_embeddings = embeddings[sample_idx]

# Cosine similarity matrix
sim = util.cos_sim(sample_embeddings, sample_embeddings).numpy()

# Ignore self similarity
np.fill_diagonal(sim, -1)

# Upper triangle only (avoid duplicate pairs)
upper = np.triu_indices(sample_size, k=1)

scores = sim[upper]

# Top similar pairs
top_k = evaluation_config["top_k_similar_pairs"]

top_indices = np.argsort(scores)[-top_k:][::-1]

same_product = 0
same_issue = 0
same_both = 0

for t in top_indices:

    i = upper[0][t]
    j = upper[1][t]

    idx1 = sample_idx[i]
    idx2 = sample_idx[j]

    p1 = df.iloc[idx1]["Product"]
    p2 = df.iloc[idx2]["Product"]

    issue1 = df.iloc[idx1]["Issue"]
    issue2 = df.iloc[idx2]["Issue"]

    if p1 == p2:
        same_product += 1

    if issue1 == issue2:
        same_issue += 1

    if p1 == p2 and issue1 == issue2:
        same_both += 1

print(f"Top {top_k} similar pairs")
print(f"Same Product : {same_product}/{top_k} ({same_product/top_k:.2%})")
print(f"Same Issue   : {same_issue}/{top_k} ({same_issue/top_k:.2%})")
print(f"Same Both    : {same_both}/{top_k} ({same_both/top_k:.2%})")

Top 1000 similar pairs
Same Product : 928/1000 (92.80%)
Same Issue   : 664/1000 (66.40%)
Same Both    : 632/1000 (63.20%)


The previous code told you:

"Among the top 1000 most similar pairs, 92.8% have the same Product."

So now this code asks:

"For the remaining 7.2% where Product is different, are they really wrong, or are they still semantically similar?"

In [26]:
count = 0

for t in top_indices:

    i = upper[0][t]
    j = upper[1][t]

    idx1 = sample_idx[i]
    idx2 = sample_idx[j]

    if df.iloc[idx1]["Product"] != df.iloc[idx2]["Product"]:

        print("="*80)
        print("Similarity:", scores[t])

        print("\nProduct 1:", df.iloc[idx1]["Product"])
        print("Issue 1:", df.iloc[idx1]["Issue"])

        print(df.iloc[idx1]["Consumer complaint narrative"][:300])

        print("\n----------------------\n")

        print("Product 2:", df.iloc[idx2]["Product"])
        print("Issue 2:", df.iloc[idx2]["Issue"])

        print(df.iloc[idx2]["Consumer complaint narrative"][:300])

        count += 1

        if count == 10:
            break

Similarity: 0.9756

Product 1: Credit card
Issue 1: Problem when making payments
In accordance with the Fair Credit Reporting Act this creditor has violated my rights under 15 USC 1681 section 6 o 2 states I have the right to privacy ( 15 USC 1681 ) ( section 6 0 4 a section 2 ) it also states a consumer reporting agency can not furnish an account without my written instructions

----------------------

Product 2: Credit reporting or other personal consumer reports
Issue 2: Improper use of your report
In accordance with the fair credit reporting act, this creditor has violated my rights under 15 usc 1681 section 602 states I have the right to privacy. 15 usc 1681 section 604a, section 2, it also states a consumer reporting agency can not furnish an account without my written instructions. Under 1
Similarity: 0.934

Product 1: Credit reporting, credit repair services, or other personal consumer reports
Issue 1: Improper use of your report
TO WHOM IT MAY CONCERN : In accordance with the 

so in prev cell:
The model is actually correct.

The CFPB taxonomy separates them, but semantically they are almost identical.
Why is this useful?

Because a mismatch does not always mean the embedding failed.

In [27]:
import numpy as np
import random
from sentence_transformers import util

random.seed(random_seed)

sample_size = evaluation_config["embedding_pair_sample_size"]
sample_idx = random.sample(range(len(df)), sample_size)

sample_embeddings = embeddings[sample_idx]

# Cosine similarity matrix
sim = util.cos_sim(sample_embeddings, sample_embeddings).numpy()

# Ignore self similarity
np.fill_diagonal(sim, -1)

same_product = 0
same_issue = 0
same_both = 0

for i in range(sample_size):

    # nearest neighbor
    nn = np.argmax(sim[i])

    idx1 = sample_idx[i]
    idx2 = sample_idx[nn]

    product1 = df.iloc[idx1]["Product"]
    product2 = df.iloc[idx2]["Product"]

    issue1 = df.iloc[idx1]["Issue"]
    issue2 = df.iloc[idx2]["Issue"]

    if product1 == product2:
        same_product += 1

    if issue1 == issue2:
        same_issue += 1

    if product1 == product2 and issue1 == issue2:
        same_both += 1

print("="*60)
print("Nearest Neighbor Evaluation")
print("="*60)

print(f"Sample size      : {sample_size}")
print(f"Product Accuracy : {same_product/sample_size:.2%}")
print(f"Issue Accuracy   : {same_issue/sample_size:.2%}")
print(f"Both Accuracy    : {same_both/sample_size:.2%}")

Nearest Neighbor Evaluation
Sample size      : 1000
Product Accuracy : 74.30%
Issue Accuracy   : 45.40%
Both Accuracy    : 43.00%


This next code is a Nearest Neighbor qualitative check.

The idea:

For each complaint, find the single most similar complaint according to the embedding, then inspect if they are actually related.

In [8]:
for i in range(10):

    nn = np.argmax(sim[i])

    idx1 = sample_idx[i]
    idx2 = sample_idx[nn]

    similarity = sim[i, nn]

    print("="*100)
    print(f"Similarity: {similarity:.4f}")

    print("\nProduct:", df.iloc[idx1]["Product"])
    print("Issue:", df.iloc[idx1]["Issue"])

    print(df.iloc[idx1]["Consumer complaint narrative"][:250])

    print("\nNearest Neighbor\n")

    print("Product:", df.iloc[idx2]["Product"])
    print("Issue:", df.iloc[idx2]["Issue"])

    print(df.iloc[idx2]["Consumer complaint narrative"][:250])

Similarity: 0.6528

Product: Checking or savings account
Issue: Managing an account
Current bank closed my account for suspected suspicious activity. I was asked to verify my account and when I did they closed my account with over {$5000.00} in it and never sent me a check for the remainder of my funds. This took place in 2022 and I

Nearest Neighbor

Product: Checking or savings account
Issue: Closing an account
My account was closed randomly they said it was because I violated terms of service but I dont see how when all I do is send money to loved ones, buy groceries, fast food, and order clothes online. I let my cousins order something off my card last we
Similarity: 0.6743

Product: Credit reporting or other personal consumer reports
Issue: Incorrect information on your report
I requested the correction of inaccurate information which another credit reporting agency found to be inaccurate and had removed. Due to your lack of response, or a response that I classify as frivolous, Cr